# 🌈 从零体验自驱动实验室（Self-Driving Lab）

欢迎来到 **Raspberry Pi Pico H + AS7341 光谱传感器 + WS2812B RGB LED** Workshop！

在这个 Notebook 中，你不需要任何控制工程或贝叶斯优化背景，就能亲手走完一个真实 **自驱动实验室（SDL）** 的完整闭环：

1. **眼睛（传感器）**：AS7341 读取 `415nm – 680nm` 共 8 个波段的真实光谱指纹。
2. **手（执行器）**：WS2812B 按 `[R, G, B]` 指令发光（每个通道 0.0–1.0 共 11 档，总计 $11^3 = 1331$ 种配方）。
3. **大脑（优化算法）**：高斯过程 + 期望改进（**GP-EI 贝叶斯优化**），根据已有测量结果决定“下一次最值得测哪一个颜色”。

> 💡 **无需硬件也能全程运行！** 默认使用 `SIMULATED` 仿真模式方便全员同步上手；连接真机 Pico H 时，只需在下面修改 `MODE = "PHYSICAL"` 和你的串口号。


## 0. 环境准备与设备连接


In [1]:
import sys
from pathlib import Path
import numpy as np

# 自动定位仓库根目录
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pocket_sdl").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pocket_sdl.config import load_config
from pocket_sdl.core import CHANNELS_NM, GRID, calibration_stats, led_values, relative_stability, spectral_loss, validate_raw
from pocket_sdl.optimize import CandidatePlanner, expected_improvement, gp_posterior
from pocket_sdl.workflow import open_device
from workshop.notebook_viz import show_color_comparison, show_convergence, show_palette_spectra, show_spectrum

# ==========================================================
# 🎛️ 一键切换模式：
#   - 零硬件体验 / 彩排：MODE = "SIMULATED", PORT = None
#   - 连接真机 Pico H：  MODE = "PHYSICAL",  PORT = "/dev/cu.usbmodemXXXX" (请用 list-ports 核对)
# ==========================================================
MODE = "SIMULATED"
PORT = None

config_path = REPO_ROOT / "configs" / ("physical.json" if MODE == "PHYSICAL" else "simulated.json")
config = load_config(config_path)
device = open_device(config, PORT)
threshold = config["sensor"]["saturation_raw_exclusive"]

print(f"✅ 已连接设备 | 模式: {device.source} | 固件: {device.device_info['firmware_version']}")
print(f"   8 个光谱通道 (nm): {list(CHANNELS_NM)}")
print(f"   设备配置指纹: {device.device_info['device_fingerprint'][:16]}...")


✅ 已连接设备 | 模式: SIMULATED | 固件: simulation-1.0.0
   8 个光谱通道 (nm): [415, 445, 480, 515, 555, 590, 630, 680]
   设备配置指纹: 4477a63da54b8107...


## 1. 点亮第一颗灯，看见光的“8 通道指纹”

人眼只有红、绿、蓝三种视锥细胞，但科研级微型光谱传感器 **AS7341** 可以同时把光线拆成 `415nm（紫）` 到 `680nm（深红）` 的 8 个通道。

下面我们依次测 **暗场、纯红、纯绿、纯蓝、白光**，看看传感器眼里的 RGB 长什么样。


In [2]:
palette = [
    ("暗场 Dark", (0.0, 0.0, 0.0)),
    ("纯红 Red", (1.0, 0.0, 0.0)),
    ("纯绿 Green", (0.0, 1.0, 0.0)),
    ("纯蓝 Blue", (0.0, 0.0, 1.0)),
    ("白光 White", (1.0, 1.0, 1.0)),
]

measured_palette = []
for label, x in palette:
    raw = validate_raw(device.measure(x), threshold)
    measured_palette.append((label, x, raw))

device.off()
show_palette_spectra(measured_palette)


### ✋ 动手试一试：调一个你的专属颜色

改一改下面 `my_rgb` 的三个数字（必须在 `0.0, 0.1, ..., 1.0` 中选），运行单元格观察 8 根光谱柱子如何随配方变化！


In [3]:
# 👇 试试改成你喜欢的组合，例如 (0.9, 0.4, 0.1) 或 (0.2, 0.8, 0.7)
my_rgb = (0.8, 0.4, 0.2)

my_raw = validate_raw(device.measure(my_rgb), threshold)
device.off()

print("输入 x =", my_rgb, "| WS2812B 亮度 (0-80) =", led_values(my_rgb))
print("AS7341 原始计数 F1-F8 =", my_raw.astype(int).tolist())
show_spectrum(my_raw, title="你的专属颜色 8 通道光谱", x=my_rgb)


输入 x = (0.8, 0.4, 0.2) | WS2812B 亮度 (0-80) = [64, 32, 16]
AS7341 原始计数 F1-F8 = [186, 314, 526, 871, 896, 1085, 1279, 1153]


## 2. 为什么要先“校准”并过“质量门”？

在真实实验室中，不能直接拿传感器原始读数 `raw` 开跑优化，因为存在环境背景光、硬件增益差异和随机噪声。我们建立三项物理基准：

1. **暗场均值 $d$**：关灯测 3 次，扣掉环境背景与暗电流。
2. **参考尺度 $A = \|w - d\|_2$**：白光参考测 3 次，将光谱归一化到无量纲向量 $z(x) = (\text{raw}(x) - d)/A$，同时要求信噪比 $A \ge 10 \times \sigma_{\text{dark}}$。
3. **固定点重复性**：在目标点连续测 5 次，确认相对波动 $< 5\%$，确保硬件没有漏光或松动。


In [4]:
cal_cfg = config["calibration"]
target_x = tuple(cal_cfg["target_x"])  # 默认 [0.3, 0.6, 0.4]，也可以改成你自己的目标

dark_rows = [validate_raw(device.measure((0, 0, 0)), threshold) for _ in range(cal_cfg["dark_repeats"])]
ref_rows = [validate_raw(device.measure(cal_cfg["reference_x"]), threshold) for _ in range(cal_cfg["reference_repeats"])]
stats = calibration_stats(dark_rows, ref_rows)

target_rows = [validate_raw(device.measure(target_x), threshold) for _ in range(cal_cfg["target_repeats"])]
target_raw = np.mean(np.stack(target_rows), axis=0)
target_z = (target_raw - stats["dark_mean"]) / stats["scale"]

stab_rows = [validate_raw(device.measure(target_x), threshold) for _ in range(cal_cfg["stability_repeats"])]
stability = relative_stability(stab_rows, stats["dark_mean"])
if stability["rms_relative"] >= cal_cfg["stability_rms_relative_max"]:
    device.off()
    raise RuntimeError("稳定性检查失败：固定位置并遮光后重新校准")
device.off()

print(f"✅ 校准通过！")
print(f"   参考尺度 A       = {stats['scale']:.2f}  (信噪比 {stats['scale'] / stats['dark_noise']:.1f}x)")
print(f"   5次固定点波动 RMS = {stability['rms_relative']:.2%}  (要求 < {cal_cfg['stability_rms_relative_max']:.0%})")
show_spectrum(target_z, title="目标光谱轮廓 z*（接下来我们要靠实验把它找出来！）", is_norm=True)


✅ 校准通过！
   参考尺度 A       = 3869.68  (信噪比 695.0x)
   5次固定点波动 RMS = 0.33%  (要求 < 5%)


## 3. 人类直觉挑战：你能几步调出目标光谱？

假设现在只给你看上面那条 **目标光谱轮廓 $z^*$**，不知道背后的目标 RGB，我们用无量纲谱差衡量有多接近：

$$L(x) = \|z(x) - z^*\|_2$$

$L(x)$ 越接近 `0`，说明复刻得越准！试着修改下面 `human_guesses` 里的 8 次猜想，看看人类靠直觉能把 $L(x)$ 压到多低。


In [5]:
# 👇 你可以修改这 8 组猜测（每个分量取 0.0 ~ 1.0，步长 0.1），再重新运行！
human_guesses = [
    (0.5, 0.5, 0.5),
    (0.2, 0.7, 0.5),
    (0.4, 0.6, 0.3),
    (0.3, 0.8, 0.4),
    (0.2, 0.6, 0.4),
    (0.4, 0.5, 0.4),
    (0.3, 0.7, 0.3),
    (0.3, 0.5, 0.5),
]

human_best_curve = []
human_best_loss = float("inf")
human_best_x = None
human_best_z = None

for step, guess in enumerate(human_guesses, 1):
    raw = validate_raw(device.measure(guess), threshold)
    loss = spectral_loss(raw, stats["dark_mean"], stats["scale"], target_raw)
    if loss < human_best_loss:
        human_best_loss = loss
        human_best_x = guess
        human_best_z = (raw - stats["dark_mean"]) / stats["scale"]
    human_best_curve.append(human_best_loss)
    print(f"第 {step:2d} 次 | x={guess} | 当前 L={loss:.4f} | 迄今最佳={human_best_loss:.4f}")

device.off()
show_spectrum(
    human_best_z,
    overlay=target_z,
    title=f"人类最佳匹配（彩柱=你的最佳，白虚线=目标，L={human_best_loss:.4f}）",
    x=human_best_x,
    is_norm=True,
)


第  1 次 | x=(0.5, 0.5, 0.5) | 当前 L=0.1132 | 迄今最佳=0.1132
第  2 次 | x=(0.2, 0.7, 0.5) | 当前 L=0.0804 | 迄今最佳=0.0804
第  3 次 | x=(0.4, 0.6, 0.3) | 当前 L=0.0654 | 迄今最佳=0.0654
第  4 次 | x=(0.3, 0.8, 0.4) | 当前 L=0.0788 | 迄今最佳=0.0654
第  5 次 | x=(0.2, 0.6, 0.4) | 当前 L=0.0579 | 迄今最佳=0.0579
第  6 次 | x=(0.4, 0.5, 0.4) | 当前 L=0.0609 | 迄今最佳=0.0579
第  7 次 | x=(0.3, 0.7, 0.3) | 当前 L=0.0281 | 迄今最佳=0.0281
第  8 次 | x=(0.3, 0.5, 0.5) | 当前 L=0.0293 | 迄今最佳=0.0281


## 4. 随机搜索 vs. 贝叶斯优化（GP-EI）同台竞技

现在我们把同样的任务交给两位“AI 实验员”，各给 **24 次实验预算**：
- 公平起见，**前 4 次实验完全相同**（同一随机种子生成的初始探索点）。
- 从第 5 次开始：
  - **Random**：继续在剩下的候选里随机瞎猜。
  - **BO (GP-EI)**：每次实测完都更新高斯过程代理模型，自动挑选期望改进（Expected Improvement）最大的下一个 RGB 配方！


In [6]:
def run_strategy(strategy_name, budget=24):
    run_cfg = load_config(config_path)
    run_cfg["measurement"]["budget"] = budget
    planner = CandidatePlanner(run_cfg, strategy_name)
    observed_x, observed_loss, best_curve = [], [], []
    best_loss, best_x, best_z = float("inf"), None, None

    for it in range(budget):
        idx = planner.choose(it, observed_x, observed_loss)
        x = tuple(GRID[idx].tolist())
        raw = validate_raw(device.measure(x), threshold)
        loss = spectral_loss(raw, stats["dark_mean"], stats["scale"], target_raw)
        observed_x.append(list(x))
        observed_loss.append(loss)
        if loss < best_loss:
            best_loss = loss
            best_x = x
            best_z = (raw - stats["dark_mean"]) / stats["scale"]
        best_curve.append(best_loss)
    device.off()
    return {
        "strategy": strategy_name,
        "observed_x": observed_x,
        "observed_loss": observed_loss,
        "best_curve": best_curve,
        "best_loss": best_loss,
        "best_x": best_x,
        "best_z": best_z,
    }

random_res = run_strategy("random", budget=24)
bo_res = run_strategy("bo", budget=24)

print(f"🎲 Random 最佳: x={random_res['best_x']} | L={random_res['best_loss']:.5f}")
print(f"🧠 BO     最佳: x={bo_res['best_x']} | L={bo_res['best_loss']:.5f}")

show_convergence({
    "Human (人类)": {"color": "#34d399", "best_so_far": human_best_curve},
    "Random (随机)": {"color": "#fb923c", "best_so_far": random_res["best_curve"]},
    "BO (GP-EI)": {"color": "#38bdf8", "best_so_far": bo_res["best_curve"]},
})

show_color_comparison([
    ("真实目标 Target", target_x, f"x={list(target_x)}"),
    ("人类最佳 Human", human_best_x, f"L={human_best_loss:.4f}"),
    ("随机最佳 Random", random_res["best_x"], f"L={random_res['best_loss']:.4f}"),
    ("BO 最佳 GP-EI", bo_res["best_x"], f"L={bo_res['best_loss']:.4f}"),
])

show_spectrum(
    bo_res["best_z"],
    overlay=target_z,
    title=f"BO 最佳复刻光谱 vs 目标轮廓（L={bo_res['best_loss']:.4f}）",
    x=bo_res["best_x"],
    is_norm=True,
)


🎲 Random 最佳: x=(0.3, 0.8, 0.3) | L=0.05929
🧠 BO     最佳: x=(0.3, 0.6, 0.4) | L=0.00180


## 5. 打开黑盒：透视贝叶斯优化的“大脑”

BO 为什么比随机搜索聪明？秘密在于它同时考虑两件事：
1. **开发（Exploitation）**：模型预测误差 $\mu(x)$ 很低的地方（看起来很像目标）。
2. **探索（Exploration）**：模型不确定度 $\sigma(x)$ 很高的地方（还没测过，可能藏着惊喜）。

下面我们截取 BO 做完第 8 次实验时的状态，直接打印它在剩余 $1323$ 个候选里排名前 5 的方案！


In [7]:
k = 8
x_train = bo_res["observed_x"][:k]
y_train = bo_res["observed_loss"][:k]
used = {121 * round(p[0] * 10) + 11 * round(p[1] * 10) + round(p[2] * 10) for p in x_train}
remaining = np.array([i for i in range(len(GRID)) if i not in used], dtype=int)

opt = config["optimizer"]
mean, var, gp_stats = gp_posterior(
    x_train, y_train, GRID[remaining],
    length_scale=opt["length_scale"], noise=opt["noise"],
)
ei = expected_improvement(
    mean, var, min(y_train),
    y_mean=gp_stats["y_mean"], y_scale=gp_stats["y_scale"], xi=opt["xi"],
)

top5 = np.argsort(ei)[::-1][:5]
print(f"截至第 {k} 步，BO 当前最佳 L = {min(y_train):.4f}；GP-EI 推荐的 Top-5 下一实验点：")
for rank, pos in enumerate(top5, 1):
    cand_x = GRID[remaining[pos]].tolist()
    pred_loss = gp_stats["y_mean"] + mean[pos] * gp_stats["y_scale"]
    pred_sd = np.sqrt(max(0.0, var[pos])) * gp_stats["y_scale"]
    print(f"  #{rank}  x={[round(v,1) for v in cand_x]}  |  预测误差 μ={pred_loss:.4f} ± σ={pred_sd:.4f}  |  EI={ei[pos]:.5f}")


截至第 8 步，BO 当前最佳 L = 0.0839；GP-EI 推荐的 Top-5 下一实验点：
  #1  x=[0.0, 1.0, 0.7]  |  预测误差 μ=0.1555 ± σ=0.0922  |  EI=0.10422
  #2  x=[0.0, 1.0, 0.6]  |  预测误差 μ=0.1513 ± σ=0.0890  |  EI=0.10388
  #3  x=[0.0, 1.0, 0.2]  |  预测误差 μ=0.1687 ± σ=0.1013  |  EI=0.10306
  #4  x=[0.0, 1.0, 0.3]  |  预测误差 μ=0.1620 ± σ=0.0965  |  EI=0.10292
  #5  x=[0.0, 1.0, 0.5]  |  预测误差 μ=0.1517 ± σ=0.0887  |  EI=0.10229


## 6. 独立复测确认（Confirmation）与安全收尾

在真实物理实验中，单次测量可能因为偶然噪声显得偏低。严谨的做法是对最佳配方进行 **3 次独立复测**（不计入优化预算，也不回写模型），报告 `均值 ± 样本标准差`，最后安全关闭设备与 LED。


In [8]:
def confirm_point(label, x, repeats=3):
    losses = []
    for _ in range(repeats):
        raw = validate_raw(device.measure(x), threshold)
        losses.append(spectral_loss(raw, stats["dark_mean"], stats["scale"], target_raw))
    device.off()
    mean_l = float(np.mean(losses))
    sd_l = float(np.std(losses, ddof=1))
    print(f"{label:8s} | x={x} | 3次独立复测 L = {mean_l:.5f} ± {sd_l:.5f}")
    return mean_l, sd_l

rand_mean, _ = confirm_point("Random", random_res["best_x"])
bo_mean, _ = confirm_point("BO", bo_res["best_x"])
if rand_mean > 0:
    print(f"\n📊 本轮独立复测谱差对比：BO 相对 Random 降低了 {(1 - bo_mean / rand_mean) * 100:.1f}%")

device.close()
print("🔒 设备已安全关闭，LED 已熄灭。")


Random   | x=(0.3, 0.8, 0.3) | 3次独立复测 L = 0.05891 ± 0.00007
BO       | x=(0.3, 0.6, 0.4) | 3次独立复测 L = 0.00124 ± 0.00052

📊 本轮独立复测谱差对比：BO 相对 Random 降低了 97.9%
🔒 设备已安全关闭，LED 已熄灭。


## 7. 延伸：启动 Workshop 本地实时 Web 交互台

如果你想在投影大屏上实时拖动 RGB 滑块、用手机扫码向 Pico 队列投稿、或者组织全场观众打 **Human vs. BO 现场擂台赛**，在仓库根目录运行：

```bash
# 1) 零硬件仿真彩排模式
python workshop/server.py --mode simulated --port 8765 --open

# 2) 连接真实 Pico H，并开放同 Wi-Fi 手机扫码（请替换为你的实际串口）
python workshop/server.py --mode physical --serial /dev/cu.usbmodemXXXX --host 0.0.0.0 --port 8765 --open
```
